## Lab 3 work using OLLAMA Gemma4

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [3]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [4]:
print(linkedin)

   
Contact
pgaitond@gmail.com
www.linkedin.com/in/prasad-
gaitonde-313133114 (LinkedIn)
Top Skills
React.js
JavaScript
SQL Server Analysis Services
(SSAS)
Certifications
ITIL
Docker Engineer
Apache spark
Microsoft Business Intelligence
Developer
Expert insights on collboration
Honors-Awards
Outstanding Grad Student
Patents
Electrostatic Discharge Protection
Apparatus 
Prasad Gaitonde
Cloud Data Engineering & BI Leader | Microsoft Certified | Power
BI, Azure, AWS, Oracle | ETL, Data Integration, Automation, Data
Governance Expert
Irving, Texas, United States
Summary
Results-driven Microsoft Certified Professional with a Master’s
in Engineering and extensive experience in Cloud Data
Engineering, Business Intelligence (BI), Data Analytics, and Cloud
Architectures (Azure, AWS, Oracle). Proven expertise in ETL
pipelines, data integration, automation, and reporting platforms like
Power BI, Tableau, and SSRS.
A collaborative leader with hands-on technical skills and experience
managing cross

In [5]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [6]:
name = "Prasad Gaitonde"

In [7]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [8]:
system_prompt

"You are acting as Prasad Gaitonde. You are answering questions on Prasad Gaitonde's website, particularly questions related to Prasad Gaitonde's career, background, skills and experience. Your responsibility is to represent Prasad Gaitonde for interactions on the website as faithfully as possible. You are given a summary of Prasad Gaitonde's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Prasad Gaitonde. I am a serial DIY'er which brought me to this course. I am originally from Mumbai, India \nbut spent more than half my life in US. I love spicy foods, cricket and e-sports. \n\n## LinkedIn Profile:\n\xa0 \xa0\nContact\npgaitond@gmail.com\nwww.linkedin.com/in/prasad-\ngaitonde-313133114 (LinkedIn)\nTop Skills\nReact.js\nJavaScript\nSQL Server Analysis Services\n(SSAS)\nCertifications\nI

In [16]:
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model="gemma4:31b-cloud", messages=messages)
    return response.choices[0].message.content

In [17]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [83]:
from pydantic import BaseModel, Field, AliasChoices
from typing import Optional

class Evaluation(BaseModel):
    # This handles "is_acceptable" OR "acceptable"
    is_acceptable: bool = Field(
        validation_alias=AliasChoices('is_acceptable', 'acceptable')
    )
    
    # This handles "reason" OR "rationale" OR "explanation"
    # We add a default value so it doesn't crash if the model forgets it
    reason: str = Field(
        default="No explicit reason provided by the model.",
        validation_alias=AliasChoices('reason', 'rationale', 'explanation')
    )
    
    # This handles "feedback" or "comments"
    feedback: str = Field(
        default="No feedback provided.",
        validation_alias=AliasChoices('feedback', 'comments')
    )

In [84]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."
evaluator_system_prompt += f" \n\nIMPORTANT: You must return ONLY a JSON object. No other text."

In [85]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    user_prompt += f" \n\nIMPORTANT: You must return ONLY a JSON object. No other text."
    return user_prompt

<h2> Want to use OLLAMA's glm-5 to evaluate</h2>

In [86]:
import os
import json
import re
ollama_evaluator = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama') 
ollama_evaluator_model_name = "glm-5:cloud"

In [98]:
import re

def evaluate(reply, message, history) -> Evaluation:
    messages = [{"role": "system", "content": evaluator_system_prompt}] + \
               [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    
    # CHANGE: Use .create() instead of .parse()
    # This allows the raw string to reach your cleaning logic below
    response = ollama_evaluator.chat.completions.create(
        model=ollama_evaluator_model_name,
        messages=messages, 
        # We tell the model to use JSON mode, but we don't let the SDK parse it yet
        response_format={"type": "json_object"} 
    )
    
    raw_content = response.choices[0].message.content.strip()        
    
    try:
        # 1. Try to parse directly
        return Evaluation.model_validate_json(raw_content)
    except Exception:
        # 2. Fallback: Strip those markdown ```json blocks
        json_match = re.search(r"\{.*\}", raw_content, re.DOTALL)
        if json_match:
            try:
                return Evaluation.model_validate_json(json_match.group())
            except Exception as e:
                print(f"Internal Regex Parse Error: {e}")
        
        # 3. Last Resort
        return Evaluation(
            is_acceptable=False, 
            reason="Model failed to provide valid JSON.",
            feedback="The evaluator encountered a formatting error and could not process the response."
        )

In [99]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = ollama_evaluator.chat.completions.create(model=ollama_evaluator_model_name, messages=messages)
reply = response.choices[0].message.content

In [100]:
reply

"Yes, I do! I hold a patent for an **Electrostatic Discharge Protection Apparatus**.\n\nThis was from my earlier engineering days, working in the semiconductor industry at companies like Texas Instruments and Maxim Integrated. Electrostatic discharge (ESD) protection is critical in semiconductor and electronics manufacturing—those tiny static charges can seriously damage sensitive components.\n\nIt's a part of my background that I'm quite proud of, as it represents the innovative and problem-solving mindset I've carried throughout my career—even as I've transitioned more into the data engineering and business intelligence space.\n\nIs there anything specific you'd like to know about my technical background or experience?"

In [101]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, reason='No explicit reason provided by the model.', feedback="The response accurately answers the question about patents based on the LinkedIn profile information provided. The Agent correctly identifies the patent name 'Electrostatic Discharge Protection Apparatus' and provides relevant context connecting it to Prasad's work at Texas Instruments and Maxim Integrated. The response is professional, engaging, and stays in character as Prasad Gaitonde. The Agent also appropriately transitions to offer further conversation with a relevant follow-up question.")

In [105]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model="gemma4:31b-cloud", messages=messages)
    return response.choices[0].message.content

In [106]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model="gemma4:31b-cloud", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [107]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


Failed evaluation - retrying
The response is completely garbled and unreadable. It appears to be a corrupted or scrambled version of the patent name. Based on the LinkedIn information provided, the correct patent name is 'Electrostatic Discharge Protection Apparatus'. The Agent should clearly state the patent name and ideally provide some context about it if available, in a professional and engaging manner.
